# XAI-PII: Trajnimi, vlerësimi dhe shpjegueshmëria e RoBERTa-s (v2)
**Tema:** Explainable Detection of Personally Identifiable Information (PII) in Text using NLP Techniques

| Pjesa | Çfarë prodhon | Ku shkon në tezë |
|---|---|---|
| 1–3 | Përgatitja e të dhënave, ndarja 80/20 (seed 42), tokenizimi me dritare rrëshqitëse | 3.3, 3.6 |
| 4 | Trajnimi deri në 8 epoka me ndalim të hershëm + krahasimi me run-in 2-epokësh | 4.1, Tabela 4.1 |
| 5 | Vlerësimi anglisht + matrica e ngatërresave | Tabela 4.2, Figura 4.4 |
| 6 | Vlerësimi shqip + **analiza sistematike e gabimeve** (tipet e gabimeve, fertiliteti BPE) | Tabela 4.3, 4.5 |
| 7 | **Zgjedhja automatike e rasteve**: gabimet më të sigurta, një rast i saktë për çdo kategori, shqip dhe anglisht | 4.6 |
| 8 | XAI mbi **të gjitha rastet**: okluzion, LIME, SHAP, Integrated Gradients, vëmendja, rollout | 4.6, Figurat e rasteve |
| 9 | Vlerësimi i shpjegimeve (ERASER, me bazë rastësore) mesatarisht mbi rastet + përputhja mes metodave | 4.6, Tabela 4.5 |
| 10 | Kundërfaktikë për adresë, emër, telefon, email | 4.7 |
| 11 | Ruajtja e të gjithë numrave | Kreu IV–V |



## 0. Instalimi i bibliotekave

In [ ]:
!pip install -q transformers datasets accelerate seqeval evaluate captum shap lime

## 1. Konfigurimi

In [ ]:
import os, json, random, ast, time, math
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# ---------------- KONFIGURIMI (ndrysho vetëm këtu) ----------------
TEST_MODE = False            # True vetëm për provë të shpejtë të kodit (model i vogël, pak dokumente)
USE_DRIVE = True             # False nëse skedarët janë ngarkuar direkt në /content
MODEL_NAME = "roberta-base"  # për eksperimentin shtesë: "xlm-roberta-base"
TRAIN = True                 # False nëse modeli është trajnuar më parë dhe ruajtur në Drive
EPOCHS = 8                   # kufiri maksimal i epokave
PATIENCE = 2                 # ndalim i hershëm: sa epoka pa përmirësim të F1-it të validimit pranohen
LR = 2e-5
BATCH = 8                    # me fp16 në T4; 
WEIGHT_DECAY = 0.01
MAX_LEN = 512
STRIDE = 128                 # mbivendosja mes dritareve për dokumentet më të gjata se 512 nën-tokenë
SEED = 42
OLD_RUN = 'colab_roberta-base'   # dosja e run-it të parë (2 epoka), vetëm për krahasim
MAX_CASES = 25               # kufiri i rasteve që shpjegohen me XAI në Pjesën 8
# ------------------------------------------------------------------

if USE_DRIVE and not TEST_MODE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/XAI_PII'
else:
    BASE = os.environ.get('XAI_BASE', '/content/XAI_PII')

RUN_NAME = MODEL_NAME.replace('/', '_') + f'_ep{EPOCHS}_es' + ('_TEST' if TEST_MODE else '')
OUT = f'{BASE}/outputs/colab_{RUN_NAME}'
MODEL_DIR = f'{OUT}/model'
os.makedirs(f'{OUT}/xai', exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Pajisja:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')
print('Rezultatet ruhen në:', OUT)
RESULTS = {}   # këtu mblidhen të gjithë numrat për tezën

## 2. Leximi i dataset-it, etiketat dhe ndarja 80/20
Ndarja bëhet **saktësisht si te modelet bazë RegEx dhe CRF** (`random.seed(42)`, përzierje e indekseve, 80% trajnim), që rezultatet të jenë të krahasueshme. Kjo korrigjon rrjedhjen e të dhënave në `src/train.py` (Tabela 4.7).

In [ ]:
df = pd.read_csv(f'{BASE}/data/raw/pii_dataset.csv')
docs = [(ast.literal_eval(t), ast.literal_eval(l)) for t, l in zip(df['tokens'], df['labels'])]

labels_all = sorted({x for _, l in docs for x in l})
labels_all.remove('O'); LABELS = ['O'] + labels_all          # e njëjta renditje si src/labels.py
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print(id2label)

rng = random.Random(SEED)
idx = list(range(len(docs))); rng.shuffle(idx)
cut = int(0.8 * len(idx))
train_idx, test_idx = idx[:cut], idx[cut:]
if TEST_MODE:
    train_idx, test_idx = train_idx[:60], test_idx[:20]
train_docs = [docs[i] for i in train_idx]
test_docs = [docs[i] for i in test_idx]
json.dump({'train': train_idx, 'test': test_idx}, open(f'{OUT}/split_indices.json', 'w'))
print(f'Trajnim: {len(train_docs)} dokumente | Test: {len(test_docs)} dokumente')

## 3. Tokenizimi me rreshtim etiketash dhe dritare rrëshqitëse
- Nën-tokeni i parë i çdo fjale merr etiketën; të tjerët dhe tokenët specialë marrin `-100`.
- Dokumentet më të gjata se 512 nën-tokenë ndahen në dritare me mbivendosje 128, që **asnjë entitet të mos humbasë** nga prerja (përmirësim ndaj `truncation=True` në `src/preprocess.py`).

In [ ]:
from transformers import AutoTokenizer
tok_kwargs = {'add_prefix_space': True} if 'roberta' in MODEL_NAME and 'xlm' not in MODEL_NAME else {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME if not TEST_MODE else os.environ.get('XAI_TOKENIZER', MODEL_NAME), **tok_kwargs)

def encode_doc(words, labels=None):
    enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LEN,
                    stride=STRIDE, return_overflowing_tokens=True)
    feats = []
    for w in range(len(enc['input_ids'])):
        wids = enc.word_ids(w)
        lab, prev = [], None
        for wi in wids:
            if wi is None: lab.append(-100)
            elif wi != prev: lab.append(label2id[labels[wi]] if labels is not None else 0)
            else: lab.append(-100)
            prev = wi
        feats.append({'input_ids': enc['input_ids'][w], 'attention_mask': enc['attention_mask'][w], 'labels': lab, 'word_ids': wids})
    return feats

# Statistika e prerjes (për nënkapitullin 3.2.2)
n_long = sum(1 for w, _ in docs if len(tokenizer(w, is_split_into_words=True)['input_ids']) > MAX_LEN) if not TEST_MODE else 0
RESULTS['dokumente_mbi_512_nentokene'] = n_long
print('Dokumente që tejkalojnë 512 nën-tokenë pas BPE:', n_long, f'({100*n_long/len(docs):.1f}%)')

from datasets import Dataset
strip = lambda feats: [{k: v for k, v in f.items() if k != 'word_ids'} for f in feats]
# Set validimi = 10% e dokumenteve të trajnimit (për ndalimin e hershëm dhe zgjedhjen e epokës); testi mbetet i paprekur
val_n = max(2, len(train_docs) // 10)
fit_docs, val_docs = train_docs[:-val_n], train_docs[-val_n:]
train_ds = Dataset.from_list(strip([f for w, l in fit_docs for f in encode_doc(w, l)]))
val_ds = Dataset.from_list(strip([f for w, l in val_docs for f in encode_doc(w, l)]))
RESULTS['dokumente_trajnim_validim_test'] = [len(fit_docs), len(val_docs), len(test_docs)]
print('Dritare: trajnim', len(train_ds), '| validim', len(val_ds))

## 4. Trajnimi i modelit (deri në 8 epoka, ndalim i hershëm mbi F1-in e validimit)
- Pas çdo epoke llogaritet F1-i (seqeval, në nivel entiteti) mbi setin e validimit.
- Nëse F1-i nuk përmirësohet për `PATIENCE` epoka rresht, trajnimi ndalon.
- Në fund ngarkohet **checkpoint-i me F1-in më të lartë të validimit**, jo ai i epokës së fundit.

In [ ]:
from transformers import (AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification, EarlyStoppingCallback)
from transformers.trainer_utils import get_last_checkpoint
try:
    import evaluate as hf_evaluate
    seqeval = hf_evaluate.load('seqeval')
except Exception as e:
    seqeval = None
    print('seqeval nuk u ngarkua, përdoret F1 në nivel tokeni për validimin:', e)

def compute_metrics(p):
    preds, labs = p
    preds = np.argmax(preds, axis=2)
    tp = [[id2label[x] for x, y in zip(pr, lb) if y != -100] for pr, lb in zip(preds, labs)]
    tl = [[id2label[y] for x, y in zip(pr, lb) if y != -100] for pr, lb in zip(preds, labs)]
    if seqeval is not None:
        r = seqeval.compute(predictions=tp, references=tl)
        return {'precision': r['overall_precision'], 'recall': r['overall_recall'], 'f1': r['overall_f1']}
    flat_p = [x for s in tp for x in s]; flat_l = [x for s in tl for x in s]
    tpc = sum(1 for a, b in zip(flat_p, flat_l) if a == b and b != 'O')
    P = tpc / max(1, sum(1 for a in flat_p if a != 'O')); R = tpc / max(1, sum(1 for b in flat_l if b != 'O'))
    return {'precision': P, 'recall': R, 'f1': 2 * P * R / max(P + R, 1e-9)}

if TRAIN:
    if TEST_MODE:
        from transformers import RobertaConfig, RobertaForTokenClassification
        cfg = RobertaConfig(vocab_size=tokenizer.vocab_size + 10, hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
                            intermediate_size=128, max_position_embeddings=MAX_LEN + 2, num_labels=len(LABELS),
                            id2label=id2label, label2id=label2id, pad_token_id=tokenizer.pad_token_id)
        model = RobertaForTokenClassification(cfg)
    else:
        model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    ckpt_dir = f'{OUT}/checkpoints'
    args = TrainingArguments(
        output_dir=ckpt_dir, learning_rate=LR, per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH, num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1',
        greater_is_better=True, save_total_limit=2, logging_steps=50, fp16=(device == 'cuda'), seed=SEED, report_to='none')
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorForTokenClassification(tokenizer), compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    last_ckpt = get_last_checkpoint(ckpt_dir) if os.path.isdir(ckpt_dir) else None
    if last_ckpt: print('Vazhdohet nga checkpoint-i:', last_ckpt)
    t0 = time.time()
    train_out = trainer.train(resume_from_checkpoint=last_ckpt)
    RESULTS['humbja_mesatare_trajnimit'] = round(train_out.training_loss, 4)
    RESULTS['koha_trajnimit_min'] = round((time.time() - t0) / 60, 1)
    trainer.save_model(MODEL_DIR); tokenizer.save_pretrained(MODEL_DIR)
    hist = trainer.state.log_history
    json.dump(hist, open(f'{OUT}/log_history.json', 'w'), indent=1)

# Tabela 4.1 ndërtohet nga log-u i ruajtur (funksionon edhe me TRAIN = False)
hist = json.load(open(f'{OUT}/log_history.json'))
evals = [h for h in hist if 'eval_f1' in h]
ep_rows = []
for h in evals:
    e = h['epoch']
    tr = [x['loss'] for x in hist if 'loss' in x and e - 1 < x.get('epoch', 0) <= e + 1e-9]
    ep_rows.append({'epoka': int(round(e)), 'humbja_trajnimit': round(tr[-1], 4) if tr else None,
                    'humbja_validimit': round(h['eval_loss'], 4), 'P_validimit': round(h['eval_precision'], 4),
                    'R_validimit': round(h['eval_recall'], 4), 'F1_validimit': round(h['eval_f1'], 4)})
best = max(ep_rows, key=lambda r: r['F1_validimit'])
RESULTS['tabela_4_1'] = ep_rows
RESULTS['epokat_e_kryera'] = len(ep_rows)
RESULTS['epoka_me_e_mire'] = best['epoka']
RESULTS['ndalim_i_hershem'] = len(ep_rows) < EPOCHS
print(pd.DataFrame(ep_rows).to_string(index=False))
print(f"Epoka më e mirë: {best['epoka']} (F1 validimi {best['F1_validimit']}); u kryen {len(ep_rows)}/{EPOCHS} epoka")

fig, ax1 = plt.subplots(figsize=(7, 3.5))
st = [(h['epoch'], h['loss']) for h in hist if 'loss' in h]
if st: ax1.plot(*zip(*st), color='#2f4b7c', lw=1, label='Humbja e trajnimit')
ax1.plot([r['epoka'] for r in ep_rows], [r['humbja_validimit'] for r in ep_rows], 'o--', color='#ff7c43', label='Humbja e validimit')
ax1.set_xlabel('Epoka'); ax1.set_ylabel('Humbja')
ax2 = ax1.twinx(); ax2.plot([r['epoka'] for r in ep_rows], [r['F1_validimit'] for r in ep_rows], 's-', color='#2ca02c', label='F1 i validimit')
ax2.axvline(best['epoka'], color='grey', ls=':', lw=1); ax2.set_ylabel('F1')
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=7, loc='center right')
plt.tight_layout(); plt.savefig(f'{OUT}/f4_1_trajnimi.png', dpi=200); plt.show()

model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR, output_attentions=True, attn_implementation='eager').to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, **tok_kwargs)
id2label = {int(k): v for k, v in model.config.id2label.items()}
label2id = {v: k for k, v in id2label.items()}
print('Etiketat e ruajtura në model:', id2label)

## 5. Vlerësimi mbi test-set-in anglisht (Tabela 4.2) dhe matrica e ngatërresave
Metrikat llogariten **në nivel entiteti me përputhje strikte**, me të njëjtin kod si modelet bazë RegEx dhe CRF.

In [ ]:
@torch.no_grad()
def predict_words(words):
    """Kthen etiketën dhe vektorin e probabiliteteve për çdo fjalë (nga nën-tokeni i parë, dritarja e parë ku shfaqet)."""
    feats = encode_doc(words)
    word_prob = [None] * len(words)
    for f in feats:
        ids = torch.tensor([f['input_ids']], device=device)
        att = torch.tensor([f['attention_mask']], device=device)
        probs = model(input_ids=ids, attention_mask=att).logits.softmax(-1)[0].float().cpu().numpy()
        prev = None
        for pos, wi in enumerate(f['word_ids']):
            if wi is not None and wi != prev and word_prob[wi] is None:
                word_prob[wi] = probs[pos]
            prev = wi
    word_prob = [p if p is not None else np.eye(len(id2label))[0] for p in word_prob]
    return [id2label[int(p.argmax())] for p in word_prob], word_prob

def spans(labels):
    out, start, typ = set(), None, None
    for i, lab in enumerate(list(labels) + ['O']):
        if lab.startswith('B-') or lab == 'O' or (lab.startswith('I-') and lab[2:] != typ):
            if typ is not None: out.add((start, i, typ))
            start, typ = (i, lab[2:]) if lab != 'O' else (None, None)
    return out

def entity_metrics(golds, preds):
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(golds, preds):
        gs, ps = spans(g), spans(p)
        for s in ps: (tp if s in gs else fp)[s[2]] += 1
        for s in gs:
            if s not in ps: fn[s[2]] += 1
    res, T, F, N = {}, 0, 0, 0
    for t in sorted(set(tp) | set(fp) | set(fn)):
        P = tp[t] / (tp[t] + fp[t]) if tp[t] + fp[t] else 0.0
        R = tp[t] / (tp[t] + fn[t]) if tp[t] + fn[t] else 0.0
        res[t] = dict(P=round(P, 3), R=round(R, 3), F1=round(2*P*R/(P+R), 3) if P + R else 0.0, support=tp[t] + fn[t])
        T += tp[t]; F += fp[t]; N += fn[t]
    P = T / (T + F) if T + F else 0.0; R = T / (T + N) if T + N else 0.0
    res['MIKRO'] = dict(P=round(P, 3), R=round(R, 3), F1=round(2*P*R/(P+R), 3) if P + R else 0.0, support=T + N)
    return res

def print_thesis_table(res, title):
    print(title)
    for k, v in res.items():
        print(f"  {k:<16} {v['support']:>5}   {v['P']:.3f} / {v['R']:.3f} / {v['F1']:.3f}".replace('.', ','))

t0 = time.time()
golds_en, preds_en, probs_en = [], [], []
for w, l in test_docs:
    p, pr = predict_words(w)
    golds_en.append(l); preds_en.append(p); probs_en.append(pr)
RESULTS['koha_vleresimit_en_sek'] = round(time.time() - t0, 1)
RESULTS['tabela_4_2_roberta'] = entity_metrics(golds_en, preds_en)
print_thesis_table(RESULTS['tabela_4_2_roberta'], 'Tabela 4.2 – RoBERTa (P / R / F1):')

# Matrica e ngatërresave në nivel tokeni (parashtesat B-/I- të bashkuara)
C = ['O', 'NAME_STUDENT', 'EMAIL', 'PHONE_NUM', 'STREET_ADDRESS', 'URL_PERSONAL', 'USERNAME']
col = lambda x: x[2:] if x != 'O' else 'O'
def conf_matrix(golds, preds, fname, title):
    M = np.zeros((7, 7), dtype=int)
    for g, p in zip(golds, preds):
        for a, b in zip(g, p): M[C.index(col(a)), C.index(col(b))] += 1
    Mn = M / np.maximum(M.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(6, 5)); im = ax.imshow(Mn, cmap='Blues', vmin=0, vmax=1)
    lab = ['O', 'NAME', 'EMAIL', 'PHONE', 'ADDRESS', 'URL', 'USERNAME']
    ax.set_xticks(range(7)); ax.set_yticks(range(7)); ax.set_xticklabels(lab, rotation=35, fontsize=8); ax.set_yticklabels(lab, fontsize=8)
    for i in range(7):
        for j in range(7):
            ax.text(j, i, f'{Mn[i, j]:.3f}\n({M[i, j]})', ha='center', va='center', fontsize=6, color='white' if Mn[i, j] > 0.5 else 'black')
    ax.set_xlabel('Klasa e parashikuar'); ax.set_ylabel('Klasa e vërtetë'); ax.set_title(title, fontsize=9); plt.colorbar(im, fraction=0.046)
    plt.tight_layout(); plt.savefig(f'{OUT}/{fname}', dpi=200); plt.show()
    return M.tolist()
RESULTS['cm_roberta'] = conf_matrix(golds_en, preds_en, 'f4_4_cm_roberta.png', 'RoBERTa – test anglisht')

## 6. Vlerësimi mbi setin shqip (Tabela 4.3) dhe analiza sistematike e gabimeve
Gabimet klasifikohen në katër tipe, për të dy gjuhët:
- **FN** – entitet i humbur (i vërtetë PII → `O`)
- **FP** – alarm i rremë (`O` → PII)
- **Ngatërrim kategorie** – PII i gjetur, por me kategori të gabuar (p.sh. adresë → email)
- **Gabim kufiri** – kategoria e saktë, por B-/I- e gabuar (entiteti ndahet ose bashkohet gabim)

In [ ]:
sq = json.load(open(f'{BASE}/data/test_shqip.json', encoding='utf-8'))
golds_sq, preds_sq, probs_sq = [], [], []
for s in sq:
    p, pr = predict_words(s['tokens'])
    golds_sq.append(s['labels']); preds_sq.append(p); probs_sq.append(pr)
RESULTS['tabela_4_3_roberta'] = entity_metrics(golds_sq, preds_sq)
print_thesis_table(RESULTS['tabela_4_3_roberta'], 'Tabela 4.3 – RoBERTa mbi shqip (P / R / F1):')
RESULTS['cm_roberta_shqip'] = conf_matrix(golds_sq, preds_sq, 'f4_4b_cm_roberta_shqip.png', 'RoBERTa – test shqip (OOD)')

def error_type(g, p):
    if col(g) == 'O': return 'FP'
    if col(p) == 'O': return 'FN'
    if col(g) != col(p): return 'Ngatërrim kategorie'
    return 'Gabim kufiri'

def collect_errors(word_lists, golds, preds, probs, lang):
    rows = []
    for si, (ws, g, p, pr) in enumerate(zip(word_lists, golds, preds, probs)):
        for wi, (a, b) in enumerate(zip(g, p)):
            if a != b:
                rows.append(dict(gjuha=lang, fjalia=si, pozicioni=wi, fjala=ws[wi], e_vertete=a, parashikimi=b,
                                 tipi=error_type(a, b), besueshmeria=round(float(pr[wi].max()), 4)))
    return rows

err_sq = pd.DataFrame(collect_errors([s['tokens'] for s in sq], golds_sq, preds_sq, probs_sq, 'shqip'))
err_en = pd.DataFrame(collect_errors([w for w, _ in test_docs], golds_en, preds_en, probs_en, 'anglisht'))
err_all = pd.concat([err_en, err_sq], ignore_index=True)
if len(err_all):
    tip = err_all.groupby(['gjuha', 'tipi']).size().unstack(fill_value=0)
    print('\nGabimet në nivel fjale sipas tipit:'); print(tip.to_string())
    RESULTS['gabimet_sipas_tipit'] = tip.to_dict()
    tip.T.plot(kind='bar', figsize=(7, 3.2), color=['#2f4b7c', '#ff7c43'])
    plt.ylabel('Numri i fjalëve'); plt.xticks(rotation=0, fontsize=8); plt.title('Tipet e gabimeve: anglisht (brenda shpërndarjes) kundrejt shqip (OOD)', fontsize=9)
    plt.tight_layout(); plt.savefig(f'{OUT}/f4_12_tipet_gabimeve.png', dpi=200); plt.show()
RESULTS['gabimet_shqip'] = err_sq.to_dict('records')
print('\nTë gjitha gabimet në setin shqip:'); print(err_sq.to_string() if len(err_sq) else '—')

# Ngatërrimet e kategorive: kush ngatërrohet me kë, dhe për cilat fjalë
if len(err_all):
    conf = err_all[err_all.tipi == 'Ngatërrim kategorie']
    if len(conf):
        pairs = conf.assign(cift=conf.e_vertete.map(col) + ' → ' + conf.parashikimi.map(col)).groupby(['gjuha', 'cift'])['fjala'].agg(['count', lambda x: ', '.join(list(dict.fromkeys(x))[:6])])
        pairs.columns = ['numri', 'shembuj']; print('\nNgatërrimet e kategorive:'); print(pairs.to_string())
        RESULTS['ngaterrimet_kategorive'] = pairs.reset_index().to_dict('records')

# Rasti "Kavajës" (Kreu 4.7): etiketa reale e modelit, me hartën e saktë nga model.config
kav = []
for si, s in enumerate(sq):
    for wi, w in enumerate(s['tokens']):
        if w.lower().startswith('kavaj'):
            pr = probs_sq[si][wi]; top = np.argsort(pr)[::-1][:3]
            kav.append({'fjalia': si + 1, 'fjala': w, 'e_vertete': s['labels'][wi], 'parashikimi': id2label[int(top[0])],
                        'top3': [(id2label[int(k)], round(float(pr[k]), 4)) for k in top]})
RESULTS['kavaja'] = kav
print('\nVerifikimi i "Kavajës":'); print(pd.DataFrame(kav).to_string() if kav else '—')

### 6.1 A lidhen gabimet me fragmentimin BPE? (hipoteza e nënkapitullit 3.6)
Për çdo fjalë që i përket një entiteti PII llogaritet numri i nën-tokenëve dhe nëse kategoria u gjet saktë.

In [ ]:
_fert_cache = {}
def fertility(w):
    if w not in _fert_cache:
        _fert_cache[w] = len(tokenizer([w], is_split_into_words=True, add_special_tokens=False)['input_ids'])
    return _fert_cache[w]

fert_rows = []
for lang, wls, gs, ps in [('anglisht', [w for w, _ in test_docs], golds_en, preds_en), ('shqip', [s['tokens'] for s in sq], golds_sq, preds_sq)]:
    for ws, g, p in zip(wls, gs, ps):
        for w, a, b in zip(ws, g, p):
            if col(a) != 'O':
                fert_rows.append(dict(gjuha=lang, kategoria=col(a), nentokene=fertility(w), sakte=int(col(a) == col(b))))
fert = pd.DataFrame(fert_rows)
fert['grupi'] = fert.nentokene.clip(upper=5).map(lambda n: '5+' if n >= 5 else str(n))
ft = fert.groupby(['gjuha', 'grupi']).agg(fjale=('sakte', 'size'), saktesia=('sakte', 'mean')).round(3)
print(ft.to_string()); RESULTS['fertiliteti_kundrejt_saktesise'] = ft.reset_index().to_dict('records')
RESULTS['fertiliteti_mesatar_entiteteve'] = fert.groupby('gjuha').nentokene.mean().round(3).to_dict()
piv = ft.saktesia.unstack(0)
piv.plot(kind='bar', figsize=(7, 3.2), color=['#2f4b7c', '#ff7c43'])
plt.xlabel('Numri i nën-tokenëve BPE për fjalë'); plt.ylabel('Saktësia e kategorisë'); plt.ylim(0, 1.05); plt.xticks(rotation=0)
plt.title('Saktësia e fjalëve PII sipas fragmentimit BPE', fontsize=9); plt.tight_layout(); plt.savefig(f'{OUT}/f4_13_fertiliteti.png', dpi=200); plt.show()

## 7. Zgjedhja sistematike e rasteve për shpjegim
Rastet nuk zgjidhen me dorë, por me rregulla të paracaktuara:
1. **Gabimet më të sigurta në shqip** – për çdo tip gabimi, 2 fjalët (nga fjali të ndryshme) ku modeli ishte më i sigurt te etiketa e gabuar. Këto janë rastet më "të vështira" dhe më informuese.
2. **Gabimet më të sigurta në anglisht** – 1 për çdo tip (a gabon modeli për të njëjtat arsye edhe brenda shpërndarjes?).
3. **Një parashikim i saktë për çdo kategori**, në shqip dhe në anglisht – si duket shpjegimi kur modeli ka të drejtë.
4. **"Kavajës"** – mbahet si një nga rastet, pa qenë rasti qendror.

Për dokumentet e gjata anglisht merret një dritare ±15 fjalë rreth fjalës, që SHAP të mbetet i përballueshëm.

In [ ]:
WIN = 15
cases = []
def add_case(words, wi, gold, pred_doc, source, kind):
    lo, hi = max(0, wi - WIN), min(len(words), wi + WIN + 1)
    EXw = [w.replace('\n', ' ').strip() or '¶' for w in words[lo:hi]]   # tokenët-hapësirë të dataset-it bëhen "¶"
    key = (tuple(EXw), wi - lo)
    if any(c['key'] == key for c in cases): return
    cases.append(dict(key=key, EX=EXw, TARGET=wi - lo, gold=gold, pred_doc=pred_doc, burimi=source, lloji=kind))

wls_en = [w for w, _ in test_docs]
for lang, E, wls, n_per in [('shqip', err_sq, [s['tokens'] for s in sq], 2), ('anglisht', err_en, wls_en, 1)]:
    if not len(E): continue
    for typ, grp in E.groupby('tipi'):
        grp = grp.sort_values('besueshmeria', ascending=False).drop_duplicates('fjalia').head(n_per)
        for _, r in grp.iterrows():
            add_case(wls[r.fjalia], int(r.pozicioni), r.e_vertete, r.parashikimi, lang, f'Gabim: {typ}')

for lang, wls, gs, ps, prs in [('shqip', [s['tokens'] for s in sq], golds_sq, preds_sq, probs_sq),
                                ('anglisht', wls_en, golds_en, preds_en, probs_en)]:
    for cat in C[1:]:
        best_c = None
        for si, (g, p, pr) in enumerate(zip(gs, ps, prs)):
            for wi, (a, b) in enumerate(zip(g, p)):
                if a == b and a == f'B-{cat}' and (best_c is None or pr[wi].max() > best_c[2]):
                    best_c = (si, wi, float(pr[wi].max()))
        if best_c:
            add_case(wls[best_c[0]], best_c[1], f'B-{cat}', f'B-{cat}', lang, f'I saktë: {cat}')

kav_pos = next(((si, wi) for si, s in enumerate(sq) for wi, w in enumerate(s['tokens']) if w.lower().startswith('kavaj')), None)
if kav_pos:   # vetëm një rast, shfaqja e parë ("Rrugën e Kavajës")
    si, wi = kav_pos
    add_case(sq[si]['tokens'], wi, sq[si]['labels'][wi], preds_sq[si][wi], 'shqip', 'Rasti "Kavajës"')

# renditja: gabimet shqip, gabimet anglisht, të saktat shqip, të saktat anglisht, "Kavajës"
grp_rank = lambda c: (3 if c['lloji'].startswith('Rasti') else (0 if c['lloji'].startswith('Gabim') else 2) + (c['burimi'] != 'shqip'))
cases.sort(key=grp_rank)
cases = cases[:MAX_CASES]
print(pd.DataFrame([{'#': k + 1, 'Burimi': c['burimi'], 'Lloji': c['lloji'], 'Fjala': c['EX'][c['TARGET']], 'E vërteta': c['gold'],
                     'Konteksti': ' '.join(c['EX'][max(0, c['TARGET'] - 5):c['TARGET'] + 6])} for k, c in enumerate(cases)]).to_string(index=False))

## 8. Shpjegueshmëria (XAI) mbi të gjitha rastet
Për çdo rast objekti i shpjegimit është **kontrastiv** (nënkapitulli 2.5.13): `logit[parashikimi] − logit[kontrasti]` për fjalën e synuar.
- Kur modeli **gabon**: kontrasti = etiketa e vërtetë → "pse zgjodhi këtë dhe jo të saktën?"
- Kur modeli **ka të drejtë**: kontrasti = `O` → "pse e njohu si PII?"

Metodat: Okluzion, LIME (5 ekzekutime, për qëndrueshmërinë), SHAP (Permutation, me kontroll efikasiteti), Integrated Gradients (Captum), vëmendja e shtresës së fundit, attention rollout. Fjala e synuar nuk maskohet kurrë.

In [ ]:
import shap
from lime.lime_base import LimeBase
from sklearn.metrics.pairwise import cosine_distances
from captum.attr import LayerIntegratedGradients
from scipy.stats import spearmanr

MASK = tokenizer.mask_token
special_ids = set(tokenizer.all_special_ids)

@torch.no_grad()
def score_batch(word_lists, tgt, c_pred, c_con, bs=64):
    """Për çdo variant të fjalisë: logit[c_pred] − logit[c_con] dhe P(c_pred) te fjala tgt."""
    scores, ps = [], []
    for i in range(0, len(word_lists), bs):
        chunk = word_lists[i:i + bs]
        enc = tokenizer(chunk, is_split_into_words=True, truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
        lg = model(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device)).logits.float()
        for b in range(len(chunk)):
            pos = enc.word_ids(b).index(tgt)
            scores.append(float(lg[b, pos, c_pred] - lg[b, pos, c_con]))
            ps.append(float(lg[b, pos].softmax(-1)[c_pred]))
    return np.array(scores), np.array(ps)

def lime_attr(EX, T, f, seed, n=500):
    """LIME (Ribeiro et al., 2016) me të njëjtin kampionim dhe kernel si LimeTextExplainer, por me maskim sipas indeksit të fjalës."""
    rs = np.random.RandomState(seed); D = len(EX)
    free = [j for j in range(D) if j != T]
    Z = np.ones((n, D))
    for i, sz in enumerate(rs.randint(1, len(free) + 1, n - 1), start=1):
        Z[i, rs.choice(free, sz, replace=False)] = 0
    s, _ = f([[w if (Z[i, j] > 0 or j == T) else MASK for j, w in enumerate(EX)] for i in range(n)])
    p = 1 / (1 + np.exp(-s))
    d = cosine_distances(Z, Z[:1]).ravel() * 100
    lb = LimeBase(lambda d: np.sqrt(np.exp(-(d ** 2) / 25 ** 2)), random_state=seed)
    _, exp, r2, _ = lb.explain_instance_with_data(Z, np.stack([1 - p, p], 1), d, 1, D, feature_selection='auto')
    w = np.zeros(D)
    for j, v in exp: w[j] = v
    w[T] = 0.0
    return w, r2

def explain_case(c, k):
    EX, T = c['EX'], c['TARGET']; D = len(EX)
    labs, prs = predict_words(EX)
    PRED, GOLD = labs[T], c['gold']
    CON = GOLD if PRED != GOLD else ('O' if GOLD != 'O' else id2label[int(np.argsort(prs[T])[::-1][1])])
    cp, cc = label2id[PRED], label2id[CON]
    f = lambda wl: score_batch(wl, T, cp, cc)
    base_s, base_p = f([EX])
    A, info = {}, dict(rasti=k, burimi=c['burimi'], lloji=c['lloji'], fjala=EX[T], e_vertete=GOLD, parashikimi=PRED,
                       parashikimi_ne_dokumentin_e_plote=c['pred_doc'],
                       kontrasti=CON, p_parashikimit=round(float(prs[T][cp]), 4), gabim=int(PRED != GOLD))

    # Okluzion
    s, _ = f([EX[:j] + [MASK] + EX[j + 1:] for j in range(D)])
    occ = base_s[0] - s; occ[T] = 0.0; A['Okluzion'] = occ

    # LIME
    runs = [lime_attr(EX, T, f, seed, n=500 if not TEST_MODE else 40) for seed in range(5 if not TEST_MODE else 2)]
    W = np.array([r[0] for r in runs]); A['LIME'] = W.mean(0)
    info['lime_std_mesatar'] = round(float(W.std(0).mean()), 4)
    info['lime_R2'] = round(float(np.mean([r[1] for r in runs])), 4)

    # SHAP (Permutation)
    def shap_fn(Z):
        return f([[w if (z[j] > 0.5 or j == T) else MASK for j, w in enumerate(EX)] for z in Z])[0]
    sv = shap.explainers.Permutation(shap_fn, shap.maskers.Independent(np.zeros((1, D))))(
        np.ones((1, D)), max_evals=max(1000, 2 * D + 2) if not TEST_MODE else 2 * D + 2)
    phi = np.array(sv.values[0], dtype=float); phi[T] = 0.0; A['SHAP'] = phi
    fb = float(np.array(sv.base_values).reshape(-1)[0])
    info['shap_gabimi_efikasitetit'] = round(abs(fb + phi.sum() - float(base_s[0])), 5)

    # Integrated Gradients
    fx = encode_doc(EX)[0]
    ids = torch.tensor([fx['input_ids']], device=device); att = torch.tensor([fx['attention_mask']], device=device)
    pos_t = fx['word_ids'].index(T)
    base_ids = torch.tensor([[t if t in special_ids else tokenizer.mask_token_id for t in fx['input_ids']]], device=device)
    def ig_forward(input_ids, attention_mask):
        lg = model(input_ids=input_ids, attention_mask=attention_mask).logits
        return lg[:, pos_t, cp] - lg[:, pos_t, cc]
    lig = LayerIntegratedGradients(ig_forward, model.base_model.embeddings.word_embeddings)
    model.zero_grad()
    at, delta = lig.attribute(ids, baselines=base_ids, additional_forward_args=(att,), n_steps=50 if not TEST_MODE else 8,
                              internal_batch_size=16, return_convergence_delta=True)
    ta = at.sum(-1)[0].detach().float().cpu().numpy()
    ig = np.zeros(D)
    for p_, wi in enumerate(fx['word_ids']):
        if wi is not None: ig[wi] += ta[p_]
    ig[T] = 0.0; A['Integrated Gradients'] = ig
    info['ig_gabimi_konvergjences'] = round(float(delta.abs().item()), 5)

    # Vëmendja dhe rollout
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=att, output_attentions=True)
    atts = [a[0].mean(0).float().cpu().numpy() for a in out.attentions]
    R = np.eye(atts[0].shape[0])
    for Am in atts:
        Ah = 0.5 * Am + 0.5 * np.eye(Am.shape[0]); Ah = Ah / Ah.sum(-1, keepdims=True); R = Ah @ R
    def to_words(row):
        w = np.zeros(D)
        for p_, wi in enumerate(fx['word_ids']):
            if wi is not None: w[wi] += row[p_]
        w[T] = 0.0
        return w
    A['Vëmendja (shtresa e fundit)'] = to_words(atts[-1][pos_t])
    A['Attention rollout'] = to_words(R[pos_t])
    tp_pos = [p_ for p_, wi in enumerate(fx['word_ids']) if wi == T]
    info['nentokene_te_fjales'] = len(tp_pos)
    info['vemendja_brenda_fjales_shtresa0'] = round(float(atts[0][np.ix_(tp_pos, tp_pos)].sum(1).mean()), 4)
    info['vemendja_brenda_fjales_shtresa_fundit'] = round(float(atts[-1][np.ix_(tp_pos, tp_pos)].sum(1).mean()), 4)

    # Lokaliteti: sa nga |SHAP| bie te fjalët në distancë ≤ 2 nga fjala e synuar
    near = [j for j in range(D) if j != T and abs(j - T) <= 2]
    info['shap_pjesa_lokale_pm2'] = round(float(np.abs(phi[near]).sum() / (np.abs(phi).sum() + 1e-12)), 4)
    top = [j for j in np.argsort(-phi) if j != T and phi[j] > 0][:3]
    info['fjalet_qe_shtyjne_drejt_parashikimit_SHAP'] = ', '.join(EX[j] for j in top) or '—'
    info['konteksti'] = ' '.join(EX)

    # Figura e rastit: metodat × fjalët, normalizuar në [-1, 1] për çdo metodë
    names = list(A); Mx = np.array([A[n] / (np.abs(A[n]).max() + 1e-12) for n in names])
    fig, ax = plt.subplots(figsize=(max(8, 0.42 * D), 0.45 * len(names) + 1.6))
    ax.imshow(Mx, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
    ax.set_xticks(range(D)); ax.set_xticklabels([w[:14] for w in EX], rotation=60, ha='right', fontsize=7)
    ax.get_xticklabels()[T].set_color('#d62728'); ax.get_xticklabels()[T].set_fontweight('bold')
    ax.set_title(f"Rasti {k}: '{EX[T]}' ({c['burimi']}, {c['lloji']}) — e vërteta {GOLD}, parashikimi {PRED} "
                 f"(p={info['p_parashikimit']:.2f}), kontrasti {CON}", fontsize=8)
    plt.tight_layout(); plt.savefig(f'{OUT}/xai/rasti_{k:02d}.png', dpi=200); plt.show(); plt.close('all')

    return info, {n: v.tolist() for n, v in A.items()}, (f, base_p[0])

XAI = []
for k, c in enumerate(cases, start=1):
    t0 = time.time()
    info, A, (f_case, bp) = explain_case(c, k)
    XAI.append(dict(info=info, attr=A, case=c, f=f_case, base_p=bp))
    print(f"[{k}/{len(cases)}] '{info['fjala']}' {info['e_vertete']} → {info['parashikimi']} | SHAP: {info['fjalet_qe_shtyjne_drejt_parashikimit_SHAP']} | {time.time() - t0:.0f}s")

xai_tbl = pd.DataFrame([x['info'] for x in XAI])
print(xai_tbl.drop(columns=['konteksti']).to_string(index=False))
RESULTS['xai_rastet'] = xai_tbl.to_dict('records')
json.dump([{'info': x['info'], 'fjalet': x['case']['EX'], 'atribuimet': x['attr']} for x in XAI],
          open(f'{OUT}/xai/atribuimet_te_gjitha_rastet.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

## 9. Vlerësimi i shpjegimeve mbi të gjitha rastet (ERASER) dhe përputhja mes metodave
- **Gjithëpërfshirja** = P(parashikimi | x) − P(parashikimi | x pa K fjalët më të rëndësishme). Më e lartë = shpjegim më besnik.
- **Mjaftueshmëria** = P(parashikimi | x) − P(parashikimi | vetëm K fjalët + fjala e synuar). Më e ulët = më mirë.
- **Bazë rastësore**: K fjalë të zgjedhura rastësisht (mesatare mbi 10 zgjedhje). Një metodë vlen vetëm nëse e kalon këtë bazë.
- Rezultatet raportohen si mesatare ± devijim standard mbi të gjitha rastet, dhe veçmas për rastet e gabuara/të sakta.

In [ ]:
K = 3
rng_e = np.random.RandomState(SEED)
er_rows = []
for x in XAI:
    EX, T, f, bp = x['case']['EX'], x['case']['TARGET'], x['f'], x['base_p']
    others = [j for j in range(len(EX)) if j != T]; k_ = min(K, len(others))
    def comp_suff(sel):
        xm = [MASK if j in sel else t for j, t in enumerate(EX)]
        xo = [t if (j in sel or j == T) else MASK for j, t in enumerate(EX)]
        _, pp = f([xm, xo])
        return bp - pp[0], bp - pp[1]
    for name, w in list(x['attr'].items()) + [('Rastësore', None)]:
        if w is None:
            cs = [comp_suff(set(rng_e.choice(others, k_, replace=False))) for _ in range(10)]
            cm, sm = np.mean([a for a, _ in cs]), np.mean([b for _, b in cs])
        else:
            sel = set([j for j in np.argsort(-np.abs(np.array(w))) if j != T][:k_])
            cm, sm = comp_suff(sel)
        er_rows.append(dict(rasti=x['info']['rasti'], gabim=x['info']['gabim'], burimi=x['info']['burimi'], metoda=name,
                            gjitheperfshirja=float(cm), mjaftueshmeria=float(sm)))
er = pd.DataFrame(er_rows)
order = list(XAI[0]['attr']) + ['Rastësore']
summ = er.groupby('metoda').agg(G_mes=('gjitheperfshirja', 'mean'), G_std=('gjitheperfshirja', 'std'),
                               M_mes=('mjaftueshmeria', 'mean'), M_std=('mjaftueshmeria', 'std')).reindex(order).round(4)
print(f'Tabela 4.5 – ERASER mbi {len(XAI)} raste (K={K}):'); print(summ.to_string())
by_grp = er.assign(grupi=er.gabim.map({1: 'gabime', 0: 'të sakta'}) + ' / ' + er.burimi).groupby(['metoda', 'grupi']).gjitheperfshirja.mean().unstack().reindex(order).round(4)
print('\nGjithëpërfshirja sipas grupit të rasteve:'); print(by_grp.to_string())
RESULTS['tabela_4_5_eraser'] = summ.reset_index().to_dict('records')
RESULTS['eraser_sipas_grupit'] = by_grp.reset_index().to_dict('records')
RESULTS['eraser_per_rast'] = er_rows

fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, (m, s, t) in zip(axs, [('G_mes', 'G_std', 'Gjithëpërfshirja (↑)'), ('M_mes', 'M_std', 'Mjaftueshmëria (↓)')]):
    ax.bar(range(len(summ)), summ[m], yerr=summ[s], capsize=3, color=['#bbbbbb' if n == 'Rastësore' else '#2f4b7c' for n in summ.index])
    ax.set_xticks(range(len(summ))); ax.set_xticklabels(summ.index, rotation=35, ha='right', fontsize=7); ax.set_title(t, fontsize=9)
    ax.axhline(0, color='k', lw=0.5)
plt.suptitle(f'Besnikëria e shpjegimeve mbi {len(XAI)} raste (K={K})', fontsize=10); plt.tight_layout()
plt.savefig(f'{OUT}/f4_11_eraser.png', dpi=200); plt.show()

# Përputhja mes metodave: Spearman mbi |atribuimin|, mesatare mbi rastet
names = list(XAI[0]['attr'])
corr = np.zeros((len(names), len(names)))
for x in XAI:
    T = x['case']['TARGET']; keep = [j for j in range(len(x['case']['EX'])) if j != T]
    for i, a in enumerate(names):
        for j, b in enumerate(names):
            r = spearmanr(np.abs(np.array(x['attr'][a])[keep]), np.abs(np.array(x['attr'][b])[keep])).correlation
            corr[i, j] += (0.0 if np.isnan(r) else r) / len(XAI)
RESULTS['korrelacioni_metodave'] = {'metodat': names, 'spearman_mesatar': np.round(corr, 3).tolist()}
plt.figure(figsize=(6, 5)); plt.imshow(corr, cmap='RdBu', vmin=-1, vmax=1)
plt.xticks(range(len(names)), names, rotation=40, ha='right', fontsize=7); plt.yticks(range(len(names)), names, fontsize=7)
for i in range(len(names)):
    for j in range(len(names)): plt.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', fontsize=7)
plt.title(f'Përputhja mes metodave (Spearman mesatar mbi {len(XAI)} raste)', fontsize=9); plt.tight_layout()
plt.savefig(f'{OUT}/f4_14_perputhja.png', dpi=200); plt.show()

# Përmbledhje e treguesve të brendshëm të XAI
agg = xai_tbl.groupby('burimi')[['lime_std_mesatar', 'shap_gabimi_efikasitetit', 'ig_gabimi_konvergjences', 'shap_pjesa_lokale_pm2',
                                 'nentokene_te_fjales', 'vemendja_brenda_fjales_shtresa0', 'vemendja_brenda_fjales_shtresa_fundit']].mean().round(4)
print('\nTreguesit mesatarë sipas gjuhës:'); print(agg.to_string())
RESULTS['xai_treguesit_sipas_gjuhes'] = agg.reset_index().to_dict('records')

## 10. Kundërfaktikë për katër kategori
Për çdo kategori ndërtohen variante të kontrolluara të së njëjtës fjali (ndryshohet vetëm një faktor: gjuha e emrit/toponimit, struktura e fjalisë, formati).
Matet P(kategoria) = P(B-) + P(I-) te fjala e synuar. Kjo teston drejtpërdrejt hipotezat që dalin nga shpjegimet e Pjesës 8.

In [ ]:
CF_GROUPS = {
    'STREET_ADDRESS': [
        ('Origjinali shqip',                "Jetoj në Rrugën e Kavajës , Tiranë .",          'Kavajës'),
        ('Toponim anglisht, fjali shqip',   "Jetoj në Rrugën e Lincoln , Tiranë .",          'Lincoln'),
        ('Renditje angleze + numër',        "Jetoj në 12 Kavajës Street , Tiranë .",         'Kavajës'),
        ('Fjali tërësisht angleze',         "I live at 12 Kavaja Street , Tirana .",         'Kavaja'),
        ('Pa nyjën "e"',                    "Jetoj në Rrugën Kavajës , Tiranë .",            'Kavajës'),
        ('Bulevard, qytet tjetër',          "Banoj në Bulevardin Dyrrah nr. 5 , Durrës .",   'Dyrrah'),
    ],
    'NAME_STUDENT': [
        ('Emër shqip, fjali shqip',         "Përshëndetje , unë jam Drilona Hoxha .",        'Drilona'),
        ('Emër anglisht, fjali shqip',      "Përshëndetje , unë jam Emily Johnson .",        'Emily'),
        ('Emër shqip, fjali angleze',       "Hello , my name is Drilona Hoxha .",            'Drilona'),
        ('Emër anglisht, fjali angleze',    "Hello , my name is Emily Johnson .",            'Emily'),
        ('Emër me shkronja të vogla',       "Përshëndetje , unë jam drilona hoxha .",        'drilona'),
        ('Emër = fjalë e zakonshme ("Bora")', "Përshëndetje , unë jam Bora Dervishi .",      'Bora'),
    ],
    'PHONE_NUM': [
        ('Format lokal shqip',              "Numri im është 0691122334 .",                   '0691122334'),
        ('Format ndërkombëtar +355',        "Numri im është +355691122334 .",                '+355691122334'),
        ('I ndarë me hapësira',             "Numri im është 069 112 2334 .",                 '069'),
        ('Format amerikan',                 "Numri im është (555) 123-4567 .",               '(555)'),
        ('I njëjti numër, fjali angleze',   "My phone number is 0691122334 .",               '0691122334'),
        ('Pa fjalën "numri"',               "Më telefononi te 0691122334 .",                 '0691122334'),
    ],
    'EMAIL': [
        ('Email me domen .com',             "Më shkruani në drilona.hoxha@gmail.com .",      'drilona.hoxha@gmail.com'),
        ('Email me domen .al',              "Më shkruani në d.hoxha@upt.al .",               'd.hoxha@upt.al'),
        ('I njëjti email, fjali angleze',   "Please write to drilona.hoxha@gmail.com .",     'drilona.hoxha@gmail.com'),
        ('Email i shkruar me fjalë',        "Më shkruani në drilona.hoxha në gmail.com .",   'drilona.hoxha'),
    ],
}
cf_rows = []
for cat, variants in CF_GROUPS.items():
    ids_cat = [label2id[l] for l in (f'B-{cat}', f'I-{cat}') if l in label2id]
    for name, sent, tw in variants:
        ws = sent.split(); t = ws.index(tw)
        labs, prs = predict_words(ws)
        cf_rows.append({'Kategoria': cat, 'Varianti': name, 'Fjala': tw, 'Parashikimi': labs[t],
                        f'P(kategoria)': round(float(prs[t][ids_cat].sum()), 4),
                        'Etiketimi i plotë': ' '.join(f'{w}/{l}' for w, l in zip(ws, labs) if l != 'O') or '—'})
cf = pd.DataFrame(cf_rows); print(cf.to_string(index=False))
RESULTS['kunderfaktiket'] = cf_rows

fig, axs = plt.subplots(1, len(CF_GROUPS), figsize=(4.2 * len(CF_GROUPS), 3.4), sharex=True)
for ax, cat in zip(np.atleast_1d(axs), CF_GROUPS):
    sub = cf[cf.Kategoria == cat]
    ax.barh(range(len(sub)), sub['P(kategoria)'], color=['#2ca02c' if v >= 0.5 else '#d62728' for v in sub['P(kategoria)']])
    ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub.Varianti, fontsize=7); ax.invert_yaxis()
    ax.axvline(0.5, color='grey', ls=':', lw=1); ax.set_xlim(0, 1); ax.set_title(cat, fontsize=9)
plt.suptitle('Kundërfaktikë: probabiliteti i kategorisë së saktë te fjala e synuar', fontsize=10); plt.tight_layout()
plt.savefig(f'{OUT}/f4_15_kunderfaktiket.png', dpi=200); plt.show()

## 11. Krahasimi me run-in 2-epokësh dhe ruajtja e rezultateve
Skedari `rezultatet_per_tezen.json`, figurat `f4_*.png` dhe dosja `xai/` ruhen në Drive te `outputs/colab_roberta-base_ep8_es/`. Shkarko dosjen (pa `checkpoints/` dhe `model/`, që janë të rënda) dhe ma sill.

In [ ]:
old_path = f'{BASE}/outputs/{OLD_RUN}/rezultatet_per_tezen.json'
if os.path.exists(old_path):
    old = json.load(open(old_path, encoding='utf-8'))
    comp = []
    for key, lab in [('tabela_4_2_roberta', 'Anglisht (brenda shpërndarjes)'), ('tabela_4_3_roberta', 'Shqip (OOD)')]:
        if key in old:
            comp.append({'Seti': lab, 'F1 – 2 epoka': old[key]['MIKRO']['F1'],
                         f"F1 – ndalim i hershëm (epoka {RESULTS['epoka_me_e_mire']})": RESULTS[key]['MIKRO']['F1']})
    print(pd.DataFrame(comp).to_string(index=False)); RESULTS['krahasimi_me_2_epoka'] = comp
else:
    print('Rezultatet e run-it 2-epokësh nuk u gjetën te', old_path)

RESULTS['konfigurimi'] = dict(MODEL_NAME=MODEL_NAME, EPOCHS_MAX=EPOCHS, PATIENCE=PATIENCE, LR=LR, BATCH=BATCH, WEIGHT_DECAY=WEIGHT_DECAY,
                             MAX_LEN=MAX_LEN, STRIDE=STRIDE, SEED=SEED, n_train_docs=len(train_docs), n_test_docs=len(test_docs),
                             n_rastesh_xai=len(XAI))
def _clean(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
json.dump(RESULTS, open(f'{OUT}/rezultatet_per_tezen.json', 'w', encoding='utf-8'), indent=1, ensure_ascii=False, default=_clean)
print('U ruajt:', f'{OUT}/rezultatet_per_tezen.json')
print(sorted(os.listdir(OUT)), sorted(os.listdir(f'{OUT}/xai')))